# Task 6 — Idempotent Replay Verification

## Yêu cầu của đề

Sửa một file Python thật, reprocess đúng file đó và chứng minh Neo4j cập nhật
không trùng, MongoDB có metadata mới trong đúng một document, Spark restart dùng
checkpoint.

## Cách triển khai và lý do

[`task6/replay_single_file.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task6/replay_single_file.py) tự động:
publish baseline, publish lại cùng revision, thêm tạm một hàm vào file thật,
publish revision mới, restart Spark, rồi khôi phục file và hai database về
revision gốc. Mỗi pha chờ đến khi Neo4j có exact node/edge ID sets và MongoDB có
đúng một `_id=file_id`; không dùng count gần đúng.

`content_sha256` là revision nội dung của contract hiện tại. `file_id` chỉ phụ
thuộc repository + path nên không đổi qua hai revision.

## Output thật đã chạy


In [1]:
import json
from pathlib import Path

report = json.loads(Path("../artifacts/task6/replay_result.json").read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))


{
  "file": "src/datasets/utils/experimental.py",
  "file_id": "6c7f76928a88fa4fd8217323002eb245416159a35a66ca6559463976f8d1fc9c",
  "file_id_stable_across_revisions": true,
  "original_content_sha256": "6acb052a2ded7705891b4be5a06fba143eb2881e75612fa606bc31fe8ef677b3",
  "modified_content_sha256": "e360b90ebfe2e1281e55872ae7199d71c72c777a580f5d05c626f565ccc32482",
  "connectors": {
    "neo4j-sink-cpg-nodes": {
      "connector": "RUNNING",
      "tasks": "RUNNING"
    },
    "neo4j-sink-cpg-edges": {
      "connector": "RUNNING",
      "tasks": "RUNNING"
    }
  },
  "first_publish": {
    "expected_node_count": 57,
    "actual_node_count": 57,
    "missing_nodes": 0,
    "stale_nodes": 0,
    "expected_edge_count": 75,
    "actual_edge_count": 75,
    "missing_edges": 0,
    "stale_edges": 0,
    "mongo_id_count": 1,
    "mongo_file_id_count": 1,
    "mongo_content_sha256": "6acb052a2ded7705891b4be5a06fba143eb2881e75612fa606bc31fe8ef677b3",
    "mongo_kafka_offset": 8
  },
  "unchan

## Bằng chứng chéo

Các ảnh Neo4j, Spark và MongoDB trong Task 4–5 được chụp trong cùng stack và
cùng `file_id` của báo cáo trên. File JSON ghi offset trước/sau restart và kết
quả cleanup.

## Lỗi đã gặp và cách khắc phục

Kiểm tra count toàn cục có thể PASS dù còn ID stale. Script được đổi sang so
sánh tập ID exact, bắt buộc cả connector và connector task `RUNNING`, bắt buộc
MongoDB count theo `_id` và `file_id` đều bằng 1, đồng thời kiểm
`processed_at`/offset không đổi qua restart. `finally` luôn phục hồi file thật
và phát revision gốc để database không bị để lại ở trạng thái demo.

## Reflection

Idempotency cần kiểm ba trạng thái: cùng revision, revision mới và restart.
Cleanup cũng là một phần của test, không phải thao tác thủ công sau cùng.

## Tái lập

```bash
python task6/replay_single_file.py   --file src/datasets/utils/experimental.py
```

Kết thúc hợp lệ phải có `status: PASS`, zero missing/stale/duplicate, MongoDB
count 1, hash cleanup bằng hash gốc và offset restart không đổi.
